# 02 – Data cleaning

**Project:** The Thin Line Between Melancholy and Rumination: What Happens When We Listen to Sad Music

In `01_explore.ipynb` I looked at the raw data and decided what needs to change.
In this notebook I put these decisions into code. The result is **one clean table** with all three samples, saved in `data/processed/`.
The raw files in `data/raw/` stay unchanged.

Each step has three parts: **what I did**, the **code**, and **what I found**.

**Contents**
0. Setup
1. Load the three samples
2. Add a `sample` column
3. Create a unique `id`
4. Compare the answer values across samples
5. Combine the three samples
6. Remove columns
7. Checkbox columns: `NaN` → `0`
8. Check the scales (1–7)
9. Combine age groups
10. Set the order of categories
11. Document missing values
12. Reorder columns and save
13. Reload and check the saved file
14. Summary

## 0. Setup

**What I did:** I load pandas and set the path to the raw data.
The path starts from where this notebook is (`notebooks/`): `..` goes one folder up, then into `data/raw`.

In [76]:
from pathlib import Path   # to work with folder paths

import pandas as pd

DATA_DIR = Path("../data/raw")         # raw data (never modified)
OUT_DIR = Path("../data/processed")    # cleaned data

DATA_DIR.exists()   # True = the folder was found

True

## 1. Load the three samples

**What I did:** I load all three files. `.copy()` stores each table in one piece in memory.
Without it, pandas showed a `PerformanceWarning` when I added new columns (harmless, but it fills the output).

In [77]:
s1 = pd.read_csv(DATA_DIR / "S1_data.csv").copy()
s2 = pd.read_csv(DATA_DIR / "S2_data.csv").copy()
s3 = pd.read_csv(DATA_DIR / "S3_data.csv").copy()

print(s1.shape)
print(s2.shape)
print(s3.shape)

(1577, 131)
(414, 131)
(445, 131)


**What I found:** S1 has 1,577 rows, S2 414 rows, S3 445 rows. All have 131 columns.

## 2. Add a `sample` column

**What I did:** After combining the files I still need to know where each person comes from.
So every table gets a column `sample` with the same value in every row.
The labels follow my check in `01_explore.ipynb` (step 4): S2 is likely the Finnish quota sample and S3 the UK sample.

In [78]:
s1["sample"] = "S1_Finland_convenience"
s2["sample"] = "S2_Finland_quota"
s3["sample"] = "S3_UK_representative"

print(s1.shape)
print(s2.shape)
print(s3.shape)

(1577, 132)
(414, 132)
(445, 132)


**What I found:** Each table now has 132 columns (131 + `sample`).

## 3. Create a unique `id`

**What I did:** `subj` starts at 1 in every file, so after combining there would be three people with the number 1.
I create a new column `id` that combines the sample and the number (e.g. `S1_1`, `S2_1`, `S3_1`).
`.astype(str)` turns the number into text, because text and numbers cannot be joined with `+`.

In [79]:
s1["id"] = "S1_" + s1["subj"].astype(str)
s2["id"] = "S2_" + s2["subj"].astype(str)
s3["id"] = "S3_" + s3["subj"].astype(str)

print(s1["id"].head(3))
print(s2["id"].head(3))
print(s3["id"].head(3))

0    S1_1
1    S1_2
2    S1_3
Name: id, dtype: str
0    S2_1
1    S2_2
2    S2_3
Name: id, dtype: str
0    S3_1
1    S3_2
2    S3_3
Name: id, dtype: str


**What I found:** The ids look as expected: `S1_1`, `S2_1`, `S3_1`, ...

*Lesson:* At first I only created `id` for S1 and got a `KeyError: 'id'` when I checked S2.
A `KeyError` means the column does not exist. I read the error from the bottom and fixed the cell.

## 4. Compare the answer values across samples

**What I did:** Before combining, I checked whether the answers are spelled the same in all three samples (e.g. `Better` and not `better`).
For every text column, the loop shows all different answers per sample, sorted alphabetically.

In [80]:
text_cols = ["age", "gender", "listen", "listensad", "AGO", "CHOICE",
             "FAM", "IMPORTANCE", "DURATION", "MOODCHANGE", "PHYSCHANGE",
             "ESSENTIAL1", "ESSENTIAL2", "ESSENTIAL3"]

for col in text_cols:
    print(col)
    print("  S1:", sorted(s1[col].dropna().unique()))
    print("  S2:", sorted(s2[col].dropna().unique()))
    print("  S3:", sorted(s3[col].dropna().unique()))
    print()

age
  S1: ['18 to 24', '25 to 34', '35 to 44', '45 to 54', '55 to 64', '65 to 74']
  S2: ['18 to 24', '25 to 34', '35 to 44', '45 to 54', '55 to 64', '65 to 74']
  S3: ['18 to 24', '25 to 34', '35 to 44', '45 to 54', '55 to 64', '65 to 74', '75 or older']

gender
  S1: ['Female', 'Male']
  S2: ['Female', 'Male']
  S3: ['Female', 'Male']

listen
  S1: ['1-2/mo', '1-2/y', '1/w', 'd', 'mult./d', 'mult/w']
  S2: ['1-2/mo', '1/w', 'd', 'mult./d', 'mult/w']
  S3: []

listensad
  S1: ['Frequently', "I don't listen to sad music voluntarily", 'Often', 'Rarely', 'Sometimes']
  S2: ['Frequently', "I don't listen to sad music voluntarily", 'Often', 'Rarely', 'Sometimes']
  S3: ['Frequently', "I don't listen to sad music voluntarily", 'Often', 'Rarely', 'Sometimes']

AGO
  S1: ['1-6 mo', '11-30 days', '2-5 days', '6-10 days', '7-12 mo', 'less than day', 'over year']
  S2: ['1-6 mo', '11-30 days', '2-5 days', '6-10 days', '7-12 mo', 'less than day', 'over year']
  S3: ['1-6 mo', '11-30 days', '2-5 d

In [81]:
# How many people are in each age group in S3?
s3["age"].value_counts()

age
45 to 54       114
55 to 64       101
35 to 44       100
25 to 34        77
18 to 24        41
65 to 74        10
75 or older      2
Name: count, dtype: int64

**What I found**
- All answer values are spelled the same in S1, S2 and S3 → no spelling fixes needed.
- **`age`:** S3 has one more category, `75 or older` (2 people; `65 to 74` has 10 people). Nobody in S1 and S2 was that old, so this is not an error.
  → Decision: keep these people and combine `65 to 74` and `75 or older` into `65 or older` (step 9).
- **`listen`:** completely empty in S3. (`1-2/y` is missing in S2 only because nobody chose it.)
  → Decision: drop `listen`, for the same reason as `expert` (step 6). `listensad` is complete and more important for my question.
- *Lesson:* the output was cut off at first ("Output is truncated", `...`). I opened it as a scrollable element to check every column.

## 5. Combine the three samples

**What I did:** `pd.concat` puts the three tables under each other. `ignore_index=True` numbers the rows again from 0.
Combining does not mean mixing: thanks to `sample`, I can still look at each sample separately.

In [82]:
df = pd.concat([s1, s2, s3], ignore_index=True)

print(df.shape)
print(df["sample"].value_counts())

(2436, 133)
sample
S1_Finland_convenience    1577
S3_UK_representative       445
S2_Finland_quota           414
Name: count, dtype: int64


**What I found:** 2,436 rows and 133 columns (131 + `sample` + `id`). S1 1,577, S2 414, S3 445.

## 6. Remove columns

**What I did:** I remove the columns I decided against in `01_explore.ipynb`:
- `expert` and `listen`: 100% missing in S3
- `REASON1–REASON13`: no labels in the data release, so they cannot be interpreted

Instead of typing 13 names, I build the list of `REASON` columns with `startswith`.
`R1–R24` start with `R` but not with `REASON`, so they stay.

In [83]:
reason_cols = [c for c in df.columns if c.startswith("REASON")]
print(len(reason_cols))   # expected: 13

df = df.drop(columns=["expert", "listen"] + reason_cols)
df.shape   # expected: 133 - 2 - 13 = 118 columns

13


(2436, 118)

**What I found:** 118 columns left.

## 7. Checkbox columns: `NaN` → `0`

**What I did:** In the checkbox columns, `NaN` means *not ticked*, not *missing*.
I change them to clear numbers: **1 = ticked, 0 = not ticked**.

`startswith(("R", "FEELING", "PHYS", "MEC"))` also finds `PHYSCHANGE`, because it starts with `PHYS`.
But `PHYSCHANGE` is not a checkbox (Worse / Same / Better), so I remove it from the list.

In [84]:
checkbox_cols = [c for c in df.columns if c.startswith(("R", "FEELING", "PHYS", "MEC"))]
print(len(checkbox_cols))   # 76, because PHYSCHANGE is included

checkbox_cols.remove("PHYSCHANGE")
print(len(checkbox_cols))   # expected: 24 R + 36 FEELING + 5 PHYS + 10 MEC = 75

76
75


In [85]:
# NaN -> 0, then 1.0 -> 1 (whole numbers instead of decimals)
df[checkbox_cols] = df[checkbox_cols].fillna(0).astype(int)

# Check: no NaN left, and only the values 0 and 1
print(df[checkbox_cols].isna().sum().sum())
print(sorted(pd.unique(df[checkbox_cols].values.ravel())))

0
[np.int64(0), np.int64(1)]


**What I found:** 75 checkbox columns, no `NaN` left, only 0 and 1.
(`np.int64(0)` is just the number 0 written the way NumPy writes it.)

Limitation: "not ticked" and "skipped the question" cannot be told apart.

## 8. Check the scales (1–7)

**What I did:** For 27 questions people could only answer from 1 to 7 (`ASM1–25`, `INTENSITY`, `PLEASURE`).
I check the smallest and largest value. A wrong value (e.g. 0 or 99) would make averages wrong later.
The first `.min()` gives the smallest value per column, the second `.min()` the smallest of all.

In [86]:
asm_cols = [c for c in df.columns if c.startswith("ASM")]
scale_cols = asm_cols + ["INTENSITY", "PLEASURE"]

print(len(scale_cols))                # expected: 27
print(df[scale_cols].min().min())     # expected: 1
print(df[scale_cols].max().max())     # expected: 7

27
1.0
7.0


**What I found:** 27 scale columns, all values between 1 and 7 → nothing to fix.

## 9. Combine age groups

**What I did:** `75 or older` only exists in S3 and has just 2 people.
I combine `65 to 74` and `75 or older` into `65 or older`, so all samples have the same age groups.
`.replace()` uses a dictionary: old value on the left, new value on the right.

In [87]:
df["age"] = df["age"].replace({"65 to 74": "65 or older",
                               "75 or older": "65 or older"})

# Number of people per age group and sample
pd.crosstab(df["age"], df["sample"])

sample,S1_Finland_convenience,S2_Finland_quota,S3_UK_representative
age,,,
18 to 24,356,80,41
25 to 34,498,122,77
35 to 44,332,90,100
45 to 54,213,65,114
55 to 64,137,41,101
65 or older,41,16,12


**What I found:** `65 or older` now has 41 people in S1, 16 in S2 and 12 in S3.
S3 is clearly older than S1 (most people in S3 are 45–54, in S1 25–34).

## 10. Set the order of categories

**What I did:** pandas sorts text alphabetically (e.g. `Better`, `Same`, `Worse`, or `10-30 min` before `5-10 min`).
I set the order that makes sense for each column, from little / bad to a lot / good.
`gender` and `CHOICE` have no natural order, so they are not included.

Check: if a value is not in my list, `pd.Categorical` turns it into `NaN`.
So I count missing values before and after. `True` means no answer was lost.

In [88]:
category_order = {
    "age": ["18 to 24", "25 to 34", "35 to 44", "45 to 54", "55 to 64", "65 or older"],
    "listensad": ["I don't listen to sad music voluntarily", "Rarely", "Sometimes", "Often", "Frequently"],
    "AGO": ["less than day", "2-5 days", "6-10 days", "11-30 days", "1-6 mo", "7-12 mo", "over year"],
    "FAM": ["unfamiliar", "somewhat familiar", "familiar", "extremely familiar"],
    "IMPORTANCE": ["Not at all", "Quite Important", "Important", "Extremely Important"],
    "DURATION": ["<5min", "5-10 min", "10-30 min", "31-60 min", "60 min+"],
    "MOODCHANGE": ["Worse", "Same", "Better"],
    "PHYSCHANGE": ["Worse", "Same", "Better"],
    "ESSENTIAL1": ["Never", "Rarely", "Sometimes", "Frequently", "Very Frequently"],
    "ESSENTIAL2": ["Never", "Rarely", "Sometimes", "Frequently", "Very Frequently"],
    "ESSENTIAL3": ["Never", "Rarely", "Sometimes", "Frequently", "Very Frequently"],
}

In [89]:
missing_before = df[list(category_order)].isna().sum()

for col, order in category_order.items():
    df[col] = pd.Categorical(df[col], categories=order, ordered=True)

missing_after = df[list(category_order)].isna().sum()
print((missing_before == missing_after).all())   # True = no answer was lost

True


**What I found:** `True` → every answer fits into its list, nothing was lost.

Open question: the order of `IMPORTANCE` follows `index.md`. It is not fully clear whether "Quite Important" means less than "Important".

Note: a CSV file only stores text, so this order is lost when saving. I will copy `category_order` into `03_analysis.ipynb`.

## 11. Document missing values

**What I did:** I do not fill in missing values. I only count them per sample, so I know how many people really answered.
`.isna()` marks missing values as `True`, `.groupby(df["sample"])` splits by sample, `.sum()` counts and `.mean() * 100` gives percent.

In [90]:
gap_cols = ["MOODCHANGE", "PHYSCHANGE", "INTENSITY"]

# Number of missing values per sample
df[gap_cols].isna().groupby(df["sample"]).sum()

,MOODCHANGE,PHYSCHANGE,INTENSITY
sample,,,
S1_Finland_convenience,366,386,14
S2_Finland_quota,93,93,4
S3_UK_representative,83,73,71


In [91]:
# Share of missing values per sample in percent
(df[gap_cols].isna().groupby(df["sample"]).mean() * 100).round(1)

,MOODCHANGE,PHYSCHANGE,INTENSITY
sample,,,
S1_Finland_convenience,23.2,24.5,0.9
S2_Finland_quota,22.5,22.5,1.0
S3_UK_representative,18.7,16.4,16.0


**What I found**

| Sample | `MOODCHANGE` | `PHYSCHANGE` | `INTENSITY` |
|---|---|---|---|
| S1 | 23.2% (366) | 24.5% (386) | 0.9% (14) |
| S2 | 22.5% (93) | 22.5% (93) | 1.0% (4) |
| S3 | 18.7% (83) | 16.4% (73) | 16.0% (71) |

→ In the analysis I always report `n`, the number of people who actually answered.

## 12. Reorder columns and save

**What I did:** I move `id` and `sample` to the front, so they are the first thing you see.
Then I save the table as a CSV file. `index=False` leaves out the row numbers (0 to 2,435), because `id` identifies every person.

In [92]:
first = ["id", "sample"]
df = df[first + [c for c in df.columns if c not in first]]
df.head()

,id,sample,subj,age,gender,listensad,ASM1,ASM2,ASM3,ASM4,...,MEC6,MEC7,MEC8,MEC9,MEC10,MOODCHANGE,PHYSCHANGE,ESSENTIAL1,ESSENTIAL2,ESSENTIAL3
0,S1_1,S1_Finland_convenience,1,35 to 44,Female,Sometimes,6.0,2.0,5.0,3.0,...,1,0,0,1,0,NaN,NaN,Rarely,Rarely,Sometimes
1,S1_2,S1_Finland_convenience,2,45 to 54,Female,Often,2.0,6.0,6.0,3.0,...,1,0,0,1,0,Better,Better,Sometimes,Rarely,Very Frequently
2,S1_3,S1_Finland_convenience,3,18 to 24,Female,Sometimes,6.0,5.0,5.0,5.0,...,0,0,0,0,0,Same,Better,Frequently,Sometimes,Sometimes
3,S1_4,S1_Finland_convenience,4,25 to 34,Male,Sometimes,5.0,4.0,4.0,6.0,...,1,0,0,1,1,NaN,NaN,Never,Never,Very Frequently
4,S1_5,S1_Finland_convenience,5,25 to 34,Female,Sometimes,3.0,4.0,5.0,7.0,...,1,0,0,1,0,Same,NaN,Sometimes,Sometimes,Rarely


In [93]:
OUT_DIR.mkdir(exist_ok=True)   # create the folder if it does not exist yet

df.to_csv(OUT_DIR / "sad_music_clean.csv", index=False)

## 13. Reload and check the saved file

**What I did:** I load the saved file again and check that everything was stored correctly.

In [94]:
check = pd.read_csv(OUT_DIR / "sad_music_clean.csv")

print(check.shape)                                  # expected: (2436, 118)
print(check["id"].is_unique)                        # expected: True (every id only once)
print(check["sample"].value_counts())               # expected: 1577 / 414 / 445
print(check[checkbox_cols].isna().sum().sum())      # expected: 0

(2436, 118)
True
sample
S1_Finland_convenience    1577
S3_UK_representative       445
S2_Finland_quota           414
Name: count, dtype: int64
0


**What I found:** (fill in after running)

## 14. Summary

**Result:** `data/processed/sad_music_clean.csv` – 2,436 people, 118 columns, one row per person.

**What changed compared to the raw data**
| Change | Why |
|---|---|
| Combined S1, S2, S3 into one table | same steps for all samples, easier analysis |
| Added `sample` | to keep the samples apart (S2 = Finland quota, S3 = UK, likely) |
| Added `id` (e.g. `S1_1`) | `subj` starts at 1 in every file |
| Removed `expert`, `listen` | 100% missing in S3 |
| Removed `REASON1–13` | no labels in the data release |
| Checkbox columns: `NaN` → `0` | `NaN` meant "not ticked" |
| `age`: `65 to 74` + `75 or older` → `65 or older` | `75 or older` only in S3 and only 2 people |
| Order of categories | for correct sorting in tables and charts (lost in CSV, reused in `03_analysis`) |

**Not changed on purpose**
- Missing values in `MOODCHANGE`, `PHYSCHANGE`, `INTENSITY` stay `NaN` and are reported.
- All answer values stayed as they were (spelling was the same in all samples).

**Next:** `03_analysis.ipynb` – compare people who felt better and worse after sad music.


## AI usage in this notebook

Most of the code was written by Claude (Anthropic) based on my instructions. Claude also helped me structure the notebook and translate my notes into English.

**My own part:** deciding which columns to remove and why, noticing that `PHYSCHANGE` is not a checkbox, deciding to keep the people aged 75 or older instead of deleting them, keeping the samples separate, and checking every output. I typed and ran the cells myself and fixed my own errors (e.g. the `KeyError`).